In [4]:
%cd /content/drive/MyDrive/tinyzero-lora-grpo
!pip install -r requirements.txt

/content/drive/MyDrive/tinyzero-lora-grpo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 12.9 MB/s eta 0:00:00


In [5]:
#@title smoke test reward
%%bash
cd drive/MyDrive/tinyzero-lora-grpo/
ls
export PYTHONPATH=$(pwd)
python scripts/prepare_data.py
python scripts/train.py \
    --dummy_reward \
    --max_steps 5 \
    --report_to none \
    --output_dir outputs/smoke

data
outputs
README.md
report
requirements.txt
scripts
src
Untitled0.ipynb
Wrote /content/drive/MyDrive/tinyzero-lora-grpo/data/processed/train.parquet (489340 rows)
Wrote /content/drive/MyDrive/tinyzero-lora-grpo/data/processed/test.parquet (1024 rows)

--- sanity sample 0 ---
{
  "nums": [
    52,
    65,
    40,
    34
  ],
  "target": 78,
  "prompt": "You are solving a Countdown arithmetic puzzle.\n\nNumbers: 52, 65, 40, 34\nTarget: 78\n\nYou must use ALL of the numbers listed above, each exactly as many times as it appears in the list. You may use +, -, *, /, parentheses.\n\nRespond in the format:\n<think>\nyour reasoning here\n</think>\n<answer>arithmetic expression only, no = sign</answer>\n"
}

--- sanity sample 1 ---
{
  "nums": [
    81,
    85,
    87
  ],
  "target": 83,
  "prompt": "You are solving a Countdown arithmetic puzzle.\n\nNumbers: 81, 85, 87\nTarget: 83\n\nYou must use ALL of the numbers listed above, each exactly as many times as it appears in the list. You may 

bash: line 1: cd: drive/MyDrive/tinyzero-lora-grpo/: No such file or directory
Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 1196.49ba/s]
Generating train split: 489340 examples [00:00, 2026653.58 examples/s]
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 196.52it/s, Materializing param=model.norm.weight]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 5/5 [02:14<00:00, 26.97s/it]


In [5]:
import json
import os

# 1 & 2. 真实 completion 原文
print("========== 2. 真实 completion 原文 (1-3条) ==========")
samples_path = "/content/drive/MyDrive/tinyzero-lora-grpo/outputs/grpo_run_500/samples.json"
if os.path.exists(samples_path):
    with open(samples_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
            # 如果是列表，取前3个；如果是字典，尝试寻找 samples 键
            samples = data if isinstance(data, list) else data.get('samples', [])
            for i, sample in enumerate(samples[:3]):
                print(f"--- Sample {i+1} ---")
                # 适配不同的 key 名称 (completion 或 completions)
                comp = sample.get('completion') or sample.get('completions', [''])[0]
                print(f"[Completion]:\n{comp}")
                print("-" * 50)
        except Exception as e:
            print(f"解析 JSON 失败: {e}")
else:
    print("未找到 samples.json")

# 3. eval 指标表
print("\n========== 3. eval 指标表 ==========")
eval_path = "/content/drive/MyDrive/tinyzero-lora-grpo/outputs/eval_lora_500.json"
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("未找到 eval_lora_500.json")

# 4. 训练时 reward 曲线
print("\n========== 4. 训练时 reward 曲线 (根据前面的训练日志总结) ==========")
print("1. 训练日志显示 `rewards/countdown_reward/mean` 确实**曾经大于 0**。")
print("2. 前几步的平均 reward 如下：")
print("   - step 1-10 左右: 0.1625, 0.1812, 0.1500")
print("   - 随后开始下降: 0.05, 0.00")
print("3. 似乎没有单独剥离记录 `format_reward`，它被揉在了单一的 countdown_reward 里面，但初期的非零奖励说明格式曾经部分正确或触发了部分得分机制。")

========== 2. 真实 completion 原文 (1-3条) ==========
--- Sample 1 ---
[Completion]:
<answer>arithmetic expression only, no = sign</answer>
<answer>arithmetic expression only, no = sign</answer>

Example:
<think>19 + 35</think>
<answer>54</answer>
<answer>54</answer>
<answer>54</answer>

Hint 1:
Think about how you can combine the numbers to get close to the target number.

Hint 2:
Consider using the numbers in a way that allows you to use the +, -, *, and / operators effectively.

Hint 3:
Try using parentheses to group numbers together and create different combinations.

Hint 4:
Remember that you can use each number exactly as many times as it appears in the list.

Hint 5:
Think about how you can use the numbers to create a sequence of operations that will lead you to the target number.

Hint 6:
Consider using the numbers in a way that allows you to use the +, -, *, and / operators effectively.

Hint 7:
Try using parentheses to group numbers together and create different combinations.

Hin

In [11]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/train.py \
    --max_steps 200 \
    --report_to none \
    --output_dir outputs/grpo_lora

Parameters: total=1,544,803,840 trainable=1,089,536 (0.0705% trainable)
{'loss': '-0.01587', 'grad_norm': '0.1296', 'learning_rate': '9.55e-06', 'num_tokens': '1.719e+04', 'completions/mean_length': '337.1', 'completions/min_length': '143.4', 'completions/max_length': '512', 'completions/clipped_ratio': '0.45', 'completions/mean_terminated_length': '195.6', 'completions/min_terminated_length': '143.4', 'completions/max_terminated_length': '262.5', 'rewards/countdown_reward/mean': '0.2313', 'rewards/countdown_reward/std': '0.4101', 'reward': '0.2313', 'reward_std': '0.4101', 'frac_reward_zero_std': '0.4', 'entropy': '0.345', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '24.81', 'epoch': '2.044e-05'}
{'loss': '0.03351', 'grad_norm': '0', 'learning_rate': '9.05e-06', 'num_tokens': '3.531e+04', 'completions/mean_length': '360.9', 'completions/min_length': '121.1', 'completions/max

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 197.64it/s, Materializing param=model.norm.weight]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 200/200 [1:28:02<00:00, 26.41s/it]


In [15]:
  %%bash
  ls -lh /content/drive/MyDrive/tinyzero-lora-grpo/outputs/eval_base.json 2>/dev/null && echo "BASE EXISTS" || echo "BASE
  MISSING"
  ls -lh /content/drive/MyDrive/tinyzero-lora-grpo/outputs/eval_lora.json 2>/dev/null && echo "LORA EXISTS" || echo "LORA
  MISSING"

BASE        
MISSING
LORA        
MISSING


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
print("下载完成，已缓存")
del model

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

下载完成，已缓存


In [26]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
DATA=data/processed/test.parquet
python scripts/eval.py --model_id Qwen/Qwen2.5-1.5B --data $DATA --out outputs/eval_base.json --max_samples 50

{
  "n": 50,
  "format_rate": 0.0,
  "valid_expr_rate": 0.0,
  "solve_rate": 0.0
}


`torch_dtype` is deprecated! Use `dtype` instead!
eval base: 100%|██████████| 50/50 [09:39<00:00, 11.59s/it]


In [28]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
DATA=data/processed/test.parquet
ADAPTER=outputs/grpo_lora/final_adapter
OUT=outputs/eval_lora.json
python scripts/eval.py --model_id Qwen/Qwen2.5-1.5B --adapter $ADAPTER --data $DATA --out $OUT --max_samples 50

{
  "n": 50,
  "format_rate": 0.0,
  "valid_expr_rate": 0.0,
  "solve_rate": 0.0
}


`torch_dtype` is deprecated! Use `dtype` instead!
eval final_adapter: 100%|██████████| 50/50 [14:22<00:00, 17.25s/it]


In [29]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python -c "
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B', trust_remote_code=True)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B', torch_dtype=torch.bfloat16, device_map='auto',
trust_remote_code=True)
model = PeftModel.from_pretrained(model, 'outputs/grpo_lora/final_adapter')
model.eval()

prompt = open('data/prompts/prompt_template.txt').read().format(nums_line='3, 7, 2', target=14)
inputs = tok(prompt, return_tensors='pt').to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tok.pad_token_id)
gen = tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('=== 模型输出 ===')
print(gen)
"

=== 模型输出 ===
<answer>arithmetic expression only, no = sign</answer>
<answer>arithmetic expression only, no = sign</answer>

Think: (7 - 3) * 2
Answer: 14
Answer: 14


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 409.58it/s, Materializing param=model.norm.weight]


In [38]:
content = open("/content/drive/MyDrive/tinyzero-lora-grpo/src/data_utils.py").read()
print(content)  # verify current content first

new_content = '''"""Countdown dataset + prompt formatting (TinyZero-compatible split)."""

from __future__ import annotations

from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset

DEFAULT_DATASET = "Jiayi-Pan/Countdown-Tasks-3to4"
# Match TinyZero\'s exact index-based split (no random seed, no filter)
# countdown.py: train = raw[: TRAIN_SIZE], test = raw[TRAIN_SIZE : TRAIN_SIZE + TEST_SIZE]
TRAIN_SIZE = 327680
TEST_SIZE = 1024


def _project_root() -> Path:
    return Path(__file__).resolve().parents[1]


def default_prompt_template_path() -> Path:
    return _project_root() / "data" / "prompts" / "prompt_template.txt"


def load_prompt_template(path: Path | None = None) -> str:
    p = path or default_prompt_template_path()
    return p.read_text(encoding="utf-8")


def format_countdown_prompt(nums: list, target: int | float, template: str | None = None) -> str:
    """Fill template placeholders: {nums_line}, {target}."""
    t = template if template is not None else load_prompt_template()
    nums_line = ", ".join(str(x) for x in nums)
    return t.format(nums_line=nums_line, target=target)


def build_countdown_dataset() -> DatasetDict:
    """Exact same split as TinyZero `examples/data_preprocess/countdown.py`.

    TinyZero uses index-based selection on the raw dataset (no filter, no random seed):
        train = raw_dataset.select(range(TRAIN_SIZE))
        test  = raw_dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))
    """
    raw = load_dataset(DEFAULT_DATASET, split="train")
    assert len(raw) > TRAIN_SIZE + TEST_SIZE, (
        f"Dataset too small: {len(raw)} <= {TRAIN_SIZE + TEST_SIZE}"
    )
    train_dataset = raw.select(range(TRAIN_SIZE))
    test_dataset = raw.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))
    return DatasetDict({"train": train_dataset, "test": test_dataset})


def add_prompt_column(ds: Dataset, template: str | None = None) -> Dataset:
    tpl = template if template is not None else load_prompt_template()

    def _row(ex: dict) -> dict:
        prompt = format_countdown_prompt(ex["nums"], ex["target"], tpl)
        return {"prompt": prompt}

    return ds.map(_row)
'''

with open("/content/drive/MyDrive/tinyzero-lora-grpo/src/data_utils.py", "w") as f:
  f.write(new_content)
print("Done — data_utils.py updated")

DEFAULT_DATASET = "Jiayi-Pan/Countdown-Tasks-3to4"                                                                   
  TRAIN_SIZE = 327680                                                                                                            
  TEST_SIZE = 1024                                                                                                               
                                                                                                                                 
                                                                                                                                 
  def _project_root():                                                                                                           
      from pathlib import Path                                                                                                   
      return Path(__file__).resolve().parents[1]                                                      

In [39]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/prepare_data.py

Wrote /content/drive/MyDrive/tinyzero-lora-grpo/data/processed/train.parquet (327680 rows)
Wrote /content/drive/MyDrive/tinyzero-lora-grpo/data/processed/test.parquet (1024 rows)

--- sanity sample 0 ---
{
  "nums": [
    44,
    19,
    35
  ],
  "target": 98,
  "prompt": "You are solving a Countdown arithmetic puzzle.\n\nNumbers: 44, 19, 35\nTarget: 98\n\nYou must use ALL of the numbers listed above, each exactly as many times as it appears in the list. You may use +, -, *, /, parentheses.\n\nRespond in the format:\n<think>\nyour reasoning here\n</think>\n<answer>arithmetic expression only, no = sign</answer>\n"
}

--- sanity sample 1 ---
{
  "nums": [
    63,
    95,
    96
  ],
  "target": 64,
  "prompt": "You are solving a Countdown arithmetic puzzle.\n\nNumbers: 63, 95, 96\nTarget: 64\n\nYou must use ALL of the numbers listed above, each exactly as many times as it appears in the list. You may use +, -, *, /, parentheses.\n\nRespond in the format:\n<think>\nyour reasoning here\n<

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 1398.10ba/s]


In [40]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/train.py --max_steps 500 --report_to none --output_dir outputs/grpo_run_500

Parameters: total=1,544,803,840 trainable=1,089,536 (0.0705% trainable)
{'loss': '0.03941', 'grad_norm': '0', 'learning_rate': '9.82e-06', 'num_tokens': '1.629e+04', 'completions/mean_length': '316', 'completions/min_length': '114.1', 'completions/max_length': '495.1', 'completions/clipped_ratio': '0.4', 'completions/mean_terminated_length': '184.4', 'completions/min_terminated_length': '114.1', 'completions/max_terminated_length': '270.6', 'rewards/countdown_reward/mean': '0.1625', 'rewards/countdown_reward/std': '0.2202', 'reward': '0.1625', 'reward_std': '0.2202', 'frac_reward_zero_std': '0.7', 'entropy': '0.3525', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '23.8', 'epoch': '3.052e-05'}
{'loss': '-0.1214', 'grad_norm': '0', 'learning_rate': '9.62e-06', 'num_tokens': '3.203e+04', 'completions/mean_length': '302.1', 'completions/min_length': '72.7', 'completions/max_length'

Generating train split: 327680 examples [00:00, 1648249.48 examples/s]
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 196.16it/s, Materializing param=model.norm.weight]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 500/500 [3:28:04<00:00, 24.97s/it]


In [41]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
DATA=data/processed/test.parquet
python scripts/eval.py --model_id Qwen/Qwen2.5-1.5B --data $DATA --out outputs/eval_base.json --max_samples 200

{
  "n": 200,
  "format_rate": 0.0,
  "valid_expr_rate": 0.0,
  "solve_rate": 0.0
}


Generating train split: 1024 examples [00:00, 157978.71 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
eval base: 100%|██████████| 200/200 [41:26<00:00, 12.43s/it]


In [1]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
DATA=data/processed/test.parquet
ADAPTER=outputs/grpo_run_500/final_adapter
python scripts/eval.py --model_id Qwen/Qwen2.5-1.5B --adapter $ADAPTER --data $DATA --out outputs/eval_lora_500.json \
--max_samples 200

{
  "n": 200,
  "format_rate": 0.0,
  "valid_expr_rate": 0.0,
  "solve_rate": 0.0
}


Generating train split: 1024 examples [00:00, 53016.43 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
eval final_adapter: 100%|██████████| 200/200 [1:10:18<00:00, 21.09s/it]


In [10]:
new_template = """You are solving a Countdown arithmetic puzzle.

Numbers: {nums_line}
Target: {target}

You must use ALL of the numbers listed above, each exactly as many times as it appears in the list.
You may use +, -, *, /, parentheses.

Respond in the format:
<think>
your reasoning here
</think>
<answer>arithmetic expression only, no = sign</answer>

<think>
"""

import os
# Ensure the directory exists
os.makedirs("/content/drive/MyDrive/tinyzero-lora-grpo/data/prompts", exist_ok=True)

with open("/content/drive/MyDrive/tinyzero-lora-grpo/data/prompts/prompt_template.txt", "w") as f:
    f.write(new_template)
print("Done")

Done


In [14]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
cat scripts/eval.py

#!/usr/bin/env python3
"""Greedy (or single-sample) eval: solve rate, format rate, valid-expression rate.

Checkpoint selection rule (per project spec):
  Use --adapter to evaluate a single checkpoint, or --checkpoint_dir to scan all
  checkpoints under a directory and automatically select the one with the highest
  solve_rate on the held-out eval set.  The best checkpoint path and its metrics
  are written to --out.
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from datasets import load_dataset
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.parsing import format_ok_for_reward, parse_countdown_response
from src.reward import compute_countdown_reward


def _find_checkpoints(checkpoint_dir: Path) -> list[Path]:
    """Return all checkpoin

In [15]:
%%bash
cat /content/drive/MyDrive/tinyzero-lora-grpo/src/parsing.py


"""Extract Countdown response segments (think block + <answer>)."""

from __future__ import annotations

import re
from dataclasses import dataclass

# Keep in sync with data/prompts/prompt_template.txt (XML-style think markers)
THINK_OPEN = '<think>'
THINK_CLOSE = '</think>'

_ANSWER_RE = re.compile(r"<answer>(?P<body>.*?)</answer>", re.DOTALL | re.IGNORECASE)


@dataclass
class ParsedResponse:
    has_think_pair: bool
    think_text: str | None
    answer_raw: str | None
    answer_inner: str | None


def parse_countdown_response(text: str, think_open: str = THINK_OPEN, think_close: str = THINK_CLOSE) -> ParsedResponse:
    """Parse model output for think tags and a single <answer>...</answer> block."""
    if text is None:
        text = ""
    t = text.strip()
    has_pair = think_open in t and think_close in t
    think_text: str | None = None
    if has_pair:
        try:
            start = t.index(think_open) + len(think_open)
            end = t.index(think_close, start)
     

In [16]:
%%bash
cat /content/drive/MyDrive/tinyzero-lora-grpo/src/reward.py

"""Layered Countdown rewards for GRPO (TRL-compatible signature)."""

from __future__ import annotations

import ast
import math
import operator
from collections import Counter
from dataclasses import dataclass
from fractions import Fraction
from typing import Any

from .parsing import format_ok_for_reward, parse_countdown_response

# Layer weights (spec)
W_FORMAT = 0.5
W_EXPR = 0.5
W_MULTISET = 0.75
W_TARGET = 3.0

RESULT_TOL = 1e-5

_ALLOWED_BINOPS: dict[type[ast.AST], Any] = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
}

_ALLOWED_UNARYOPS: dict[type[ast.AST], Any] = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


@dataclass
class RewardBreakdown:
    format_score: float
    expr_score: float
    multiset_score: float
    target_score: float
    total: float
    solved: bool
    details: dict[str, Any]


def _as_number(n: ast.expr) -> Fraction | N

In [27]:
import sys
import os

parsing_code = """\n\"\"\"Extract Countdown response segments (think block + <answer>).\"\"\"\n\nfrom __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\n\nTHINK_OPEN = '<think>'\nTHINK_CLOSE = '</think>'\n\n_ANSWER_RE = re.compile(r"<answer>(?P<body>.*?)</answer>", re.DOTALL | re.IGNORECASE)\n\n@dataclass\nclass ParsedResponse:\n    has_think_pair: bool\n    think_text: str | None\n    answer_raw: str | None\n    answer_inner: str | None\n\ndef parse_countdown_response(text: str, think_open: str = THINK_OPEN, think_close: str = THINK_CLOSE) -> ParsedResponse:\n    if text is None:\n        text = ""\n    t = text.strip()\n    # Case: completion starts inside a think block (open tag in prompt)\n    has_pair = think_close in t\n    think_text: str | None = None\n    if has_pair:\n        if think_open in t:\n            try:\n                start = t.index(think_open) + len(think_open)\n                end = t.index(think_close, start)\n                think_text = t[start:end].strip()\n            except ValueError:\n                has_pair = False\n        else:\n            try:\n                end = t.index(think_close)\n                think_text = t[:end].strip()\n            except ValueError:\n                has_pair = False\n\n    m = _ANSWER_RE.search(t)\n    answer_raw = m.group(0) if m else None\n    answer_inner = m.group("body").strip() if m else None\n    return ParsedResponse(\n        has_think_pair=has_pair,\n        think_text=think_text,\n        answer_raw=answer_raw,\n        answer_inner=answer_inner,\n    )\n\ndef format_ok_for_reward(\n    parsed: ParsedResponse,\n    raw_text: str | None = None,\n    think_close: str = THINK_CLOSE,\n) -> bool:\n    if not parsed.has_think_pair or parsed.answer_inner is None:\n        return False\n    if "=" in parsed.answer_inner:\n        return False\n    if raw_text is not None:\n        try:\n            end_think = raw_text.index(think_close) + len(think_close)\n        except ValueError:\n            return False\n        if _ANSWER_RE.search(raw_text[end_think:]) is None:\n            return False\n    return True\n"""

# Save the new parsing script
with open("/content/drive/MyDrive/tinyzero-lora-grpo/src/parsing.py", "w") as f:
    f.write(parsing_code.strip())
print("Done — parsing.py updated")

# Verification logic
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Clear modules cache
for mod in list(sys.modules.keys()):
    if "parsing" in mod or "reward" in mod:
        del sys.modules[mod]

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B", torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

# Attempt to find the best available adapter
adapter_path = "/content/drive/MyDrive/tinyzero-lora-grpo/outputs/grpo_run_500/checkpoint-500"
if not os.path.exists(adapter_path):
    adapter_path = "/content/drive/MyDrive/tinyzero-lora-grpo/outputs/grpo_lora/final_adapter"

print(f"Loading adapter from: {adapter_path}")
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

template_text = open("/content/drive/MyDrive/tinyzero-lora-grpo/data/prompts/prompt_template.txt").read()
prompt = template_text.format(nums_line="3, 7, 2", target=14)
inputs = tok(prompt, return_tensors="pt").to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tok.pad_token_id)
completion = tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== COMPLETION ===")
print(completion)

sys.path.insert(0, "/content/drive/MyDrive/tinyzero-lora-grpo")
from src.parsing import parse_countdown_response, format_ok_for_reward
from src.reward import compute_countdown_reward

parsed = parse_countdown_response(completion)
print(f"\nhas_think_pair: {parsed.has_think_pair}")
print(f"answer_inner:   {parsed.answer_inner}")
print(f"format_ok:      {format_ok_for_reward(parsed, raw_text=completion)}")
bd = compute_countdown_reward(completion, [3, 7, 2], 14)
print(f"reward total:   {bd.total}  solved: {bd.solved}")

Done — parsing.py updated


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading adapter from: /content/drive/MyDrive/tinyzero-lora-grpo/outputs/grpo_run_500/checkpoint-500
=== COMPLETION ===
</think>
<answer>
</answer>

1. **Identify the numbers and the target:**
   - Numbers: 3, 7, 2
   - Target: 14

2. **Consider the operations:**
   - Addition (+)
   - Subtraction (-)
   - Multiplication (*)
   - Division (/)

3. **Start with the simplest operations:**
   - Try to use multiplication and addition to get close to 14.

4. **Combine the numbers using multiplication and addition:**
   - \(7 \times 2 = 14\)

5. **Verify the result:**
   - \(7 \times 2 = 14\)

6. **Check if the numbers are used exactly as many times as they appear in the list:**
   - 7 appears once
   - 2 appears once
   - 3 appears once

7. **Conclusion:**
   - The arithmetic expression that

has_think_pair: True
answer_inner:   
format_ok:      True
reward total:   0.5  solved: False


In [43]:
import os

sft_script = r'''#!/usr/bin/env python3
"""SFT warm-up: teach format before GRPO."""
from __future__ import annotations
import argparse, sys
from itertools import permutations, product
from fractions import Fraction
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from trl import SFTTrainer, SFTConfig
from src.data_utils import build_countdown_dataset, load_prompt_template

def solve(nums, target):
    ops = [("+", lambda a,b: a+b), ("-", lambda a,b: a-b),
          ("*", lambda a,b: a*b), ("/", lambda a,b: a/b if b!=0 else None)]
    n = len(nums)
    tmpls3 = ["({a}{o1}{b}){o2}{c}", "{a}{o1}({b}{o2}{c})"]
    tmpls4 = ["(({a}{o1}{b}){o2}{c}){o3}{d}", "({a}{o1}({b}{o2}{c})){o3}{d}",
              "({a}{o1}{b}){o2}({c}{o3}{d})", "{a}{o1}(({b}{o2}{c}){o3}{d})",
              "{a}{o1}({b}{o2}({c}{o3}{d}))"]
    for perm in permutations(nums):
        p = [str(x) for x in perm]
        if n == 3:
            a,b,c = p
            for (s1,f1),(s2,f2) in product(ops, repeat=2):
                for tmpl in tmpls3:
                    expr = tmpl.format(a=a,b=b,c=c,o1=s1,o2=s2)
                    try:
                        v = Fraction(eval(expr))
                        if abs(float(v)-target)<1e-5: return expr
                    except: pass
        elif n == 4:
            a,b,c,d = p
            for (s1,f1),(s2,f2),(s3,f3) in product(ops, repeat=3):
                for tmpl in tmpls4:
                    expr = tmpl.format(a=a,b=b,c=c,d=d,o1=s1,o2=s2,o3=s3)
                    try:
                        v = Fraction(eval(expr))
                        if abs(float(v)-target)<1e-5: return expr
                    except: pass
    return None

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", default="Qwen/Qwen2.5-1.5B")
    ap.add_argument("--num_examples", type=int, default=120)
    ap.add_argument("--output_dir", default="outputs/sft_adapter")
    ap.add_argument("--epochs", type=int, default=3)
    args = ap.parse_args()

    template = load_prompt_template()
    raw = build_countdown_dataset()["train"]

    print("Solving examples...")
    records = []
    for row in raw:
        if len(records) >= args.num_examples:
            break
        nums = list(row["nums"]); target = row["target"]
        expr = solve(nums, target)
        if expr is None: continue
        nums_line = ", ".join(str(n) for n in nums)
        prompt = template.format(nums_line=nums_line, target=target)
        completion = f"I need to use {nums_line} to reach {target}.\nAfter checking: {expr} = {target}.\n</think>\n<answer>{expr}</answer>"
        records.append({"text": prompt + completion})

    print(f"Built {len(records)} SFT examples")
    sft_ds = Dataset.from_list(records)
    tok = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        args.model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
    )
    lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")
    model = get_peft_model(model, lora_cfg)

    targs = SFTConfig(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy='no',
        report_to='none',
        dataset_text_field='text',
    )
    trainer = SFTTrainer(
        model=model,
        args=targs,
        train_dataset=sft_ds,
        processing_class=tok,
    )
    trainer.train()
    out = Path(args.output_dir) / "final_adapter"
    model.save_pretrained(out)
    tok.save_pretrained(out)
    print(f"Saved SFT adapter -> {out}")

if __name__ == "__main__":
    main()
'''

script_path = "/content/drive/MyDrive/tinyzero-lora-grpo/scripts/sft.py"
os.makedirs(os.path.dirname(script_path), exist_ok=True)
with open(script_path, "w") as f:
    f.write(sft_script)
print(f"Updated {script_path}")

Updated /content/drive/MyDrive/tinyzero-lora-grpo/scripts/sft.py


In [44]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/sft.py --num_examples 120 --epochs 3 --output_dir outputs/sft_adapter

Solving examples...
Built 120 SFT examples
{'loss': '1.87', 'grad_norm': '0.689', 'learning_rate': '0.00016', 'entropy': '1.698', 'num_tokens': '1.23e+04', 'mean_token_accuracy': '0.6081', 'epoch': '0.6667'}
{'loss': '1.462', 'grad_norm': '1.098', 'learning_rate': '0.0001156', 'entropy': '1.49', 'num_tokens': '2.473e+04', 'mean_token_accuracy': '0.6776', 'epoch': '1.333'}
{'loss': '1.051', 'grad_norm': '1.322', 'learning_rate': '7.111e-05', 'entropy': '1.146', 'num_tokens': '3.703e+04', 'mean_token_accuracy': '0.7367', 'epoch': '2'}
{'loss': '0.7623', 'grad_norm': '1.385', 'learning_rate': '2.667e-05', 'entropy': '0.7999', 'num_tokens': '4.938e+04', 'mean_token_accuracy': '0.8042', 'epoch': '2.667'}
{'train_runtime': '46.36', 'train_samples_per_second': '7.764', 'train_steps_per_second': '0.971', 'train_loss': '1.217', 'entropy': '0.6843', 'num_tokens': '5.554e+04', 'mean_token_accuracy': '0.8239', 'epoch': '3'}
Saved SFT adapter -> outputs/sft_adapter/final_adapter


`torch_dtype` is deprecated! Use `dtype` instead!
Truncating train dataset: 100%|██████████| 120/120 [00:00<00:00, 28184.37 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 45/45 [00:46<00:00,  1.03s/it]


In [45]:

import torch, sys
sys.path.insert(0, "/content/drive/MyDrive/tinyzero-lora-grpo")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from src.parsing import parse_countdown_response, format_ok_for_reward
from src.reward import compute_countdown_reward

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B", dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
model = PeftModel.from_pretrained(model,
"/content/drive/MyDrive/tinyzero-lora-grpo/outputs/sft_adapter/final_adapter")
model.eval()

template = open("/content/drive/MyDrive/tinyzero-lora-grpo/data/prompts/prompt_template.txt").read()

# 测3条
tests = [
    ([3, 7, 2], 12),   # 3+7+2=12
    ([4, 5, 6], 14),
    ([10, 2, 3, 1], 4),
]
for nums, target in tests:
    prompt = template.format(nums_line=", ".join(str(n) for n in nums), target=target)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False,
pad_token_id=tok.pad_token_id)
    completion = tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    bd = compute_countdown_reward(completion, nums, target)
    print(f"nums={nums} target={target}")
    print(f"  completion: {completion[:120]!r}")
    print(f"  format_ok={format_ok_for_reward(parse_countdown_response(completion),
raw_text=completion)}  solved={bd.solved}  reward={bd.total}")
    print()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

nums=[3, 7, 2] target=12
  completion: 'I need to find a combination of operations that will yield 12 using 3, 7, 2.\n</think>\n<answer>(3 + 7) * 2</answer>'
  format_ok=True  solved=False  reward=1.75

nums=[4, 5, 6] target=14
  completion: 'I need to find a combination of operations that will result in 14 using 4, 5, 6.\n</think>\n<answer>(6 * 5) + 4</answer>'
  format_ok=True  solved=False  reward=1.75

nums=[10, 2, 3, 1] target=4
  completion: "I need to use 10, 2, 3, 1 to get 4.\nI can't use any number more than once.\nI can use +, -, *, /, parentheses.\n</think>\n<"
  format_ok=True  solved=False  reward=1.75



In [46]:
%%bash
cat /content/drive/MyDrive/tinyzero-lora-grpo/scripts/train.py

#!/usr/bin/env python3
"""GRPO training with LoRA on Countdown (TRL >= 0.29)."""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoTokenizer, TrainerCallback, TrainerControl, TrainerState, TrainingArguments
from trl import GRPOConfig, GRPOTrainer

from src.model_utils import DEFAULT_LORA, count_params
from src.reward import countdown_reward, countdown_reward_with_breakdown, dummy_reward


class SampleLogCallback(TrainerCallback):
    """Every `log_every` steps, generate completions for a few train rows and save reward breakdowns."""

    def __init__(
        self,
        dataset,
        tokenizer,
        out_path: Path,
        log_every: int = 50,
        n_samples: int = 4,
        max_new_tokens: int = 512,
  

In [12]:
train_script = r'''#!/usr/bin/env python3
"""GRPO training with LoRA on Countdown (TRL >= 0.29)."""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (AutoTokenizer, TrainerCallback, TrainerControl, TrainerState,
                          TrainingArguments, AutoModelForCausalLM)
from trl import GRPOConfig, GRPOTrainer

from src.model_utils import DEFAULT_LORA, count_params
from src.reward import countdown_reward, countdown_reward_with_breakdown, dummy_reward

class SampleLogCallback(TrainerCallback):
    def __init__(self, dataset, tokenizer, out_path, log_every=50, n_samples=4, max_new_tokens=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.out_path = out_path
        self.log_every = log_every
        self.n_samples = n_samples
        self.max_new_tokens = max_new_tokens
        self.records = []
        out_path.parent.mkdir(parents=True, exist_ok=True)

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.log_every != 0:
            return
        if model is None:
            return
        model.eval()
        device = next(model.parameters()).device
        n = min(self.n_samples, len(self.dataset))
        prompts, nums_list, targets = [], [], []
        for i in range(n):
            row = self.dataset[i]
            prompts.append(row["prompt"])
            nums_list.append(row["nums"])
            targets.append(row["target"])
        completions = []
        for prompt in prompts:
            inputs = self.tokenizer(prompt, return_tensors="pt").to(device)
            with torch.inference_mode():
                out = model.generate(
                    **inputs, max_new_tokens=self.max_new_tokens,
                    do_sample=False, pad_token_id=self.tokenizer.pad_token_id,
                )
            gen_ids = out[0, inputs.input_ids.shape[1]:]
            completions.append(self.tokenizer.decode(gen_ids, skip_special_tokens=True))
        _, breakdowns = countdown_reward_with_breakdown(
            prompts, completions, nums=nums_list, target=targets
        )
        step_records = []
        for i, (prompt, completion, bd) in enumerate(zip(prompts, completions, breakdowns)):
            step_records.append({
                "step": state.global_step,
                "sample_idx": i,
                "prompt": prompt[:500],
                "completion": completion[:1500],
                "parsed_answer": bd.details.get("value"),
                "reward_breakdown": {
                    "format": bd.format_score,
                    "expr": bd.expr_score,
                    "multiset": bd.multiset_score,
                    "target": bd.target_score,
                    "total": bd.total,
                },
                "solved": bd.solved,
            })
        self.records.extend(step_records)
        self.out_path.write_text(
            json.dumps(self.records, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print(f"[SampleLog] step={state.global_step} solve_rate={sum(r['solved'] for r in step_records)}/{n}")
        model.train()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_id", type=str, default="Qwen/Qwen2.5-1.5B")
    ap.add_argument("--init_adapter", type=Path, default=None)
    ap.add_argument("--train_data", type=Path, default=ROOT / "data" / "processed" / "train.parquet")
    ap.add_argument("--output_dir", type=Path, default=ROOT / "outputs" / "grpo_run")
    ap.add_argument("--max_steps", type=int, default=200)
    ap.add_argument("--learning_rate", type=float, default=1e-5)
    ap.add_argument("--per_device_train_batch_size", type=int, default=4)
    ap.add_argument("--lora_r", type=int, default=8)
    ap.add_argument("--report_to", type=str, default="none")
    args = ap.parse_args()

    train_ds = load_dataset("parquet", data_files=str(args.train_data), split="train")
    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    peft_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=args.lora_r, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], bias="none",
    )

    if args.init_adapter:
        model = AutoModelForCausalLM.from_pretrained(args.model_id, torch_dtype=torch.bfloat16, device_map="auto")
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(args.init_adapter), is_trainable=True)
    else:
        model = args.model_id

    grpo_args = GRPOConfig(
        output_dir=str(args.output_dir), learning_rate=args.learning_rate, max_steps=args.max_steps,
        per_device_train_batch_size=args.per_device_train_batch_size, bf16=True,
        num_generations=4, max_completion_length=512, report_to=[args.report_to] if args.report_to != 'none' else [],
    )

    trainer = GRPOTrainer(
        model=model, reward_funcs=countdown_reward, args=grpo_args,
        train_dataset=train_ds, processing_class=tokenizer, peft_config=peft_config if not args.init_adapter else None,
    )

    stats = count_params(trainer.model)
    print(f"Parameters: total={stats.total:,} trainable={stats.trainable:,} ({stats.trainable_pct:.4f}% trainable)")

    trainer.train()
    trainer.save_model(str(args.output_dir / "final_adapter"))

if __name__ == "__main__":
    main()
'''

with open("/content/drive/MyDrive/tinyzero-lora-grpo/scripts/train.py", "w") as f:
    f.write(train_script)
print("Done")

Done


In [2]:
%%bash
# 确保安装了必要的库
pip install -q trl peft transformers datasets accelerate

cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/train.py \
    --init_adapter outputs/sft_adapter/final_adapter \
    --max_steps 500 \
    --report_to none \
    --output_dir outputs/grpo_from_sft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 59.9 MB/s eta 0:00:00
Loaded init adapter from: outputs/sft_adapter/final_adapter
Parameters: total=1,544,803,840 trainable=0 (0.0000% trainable)
{'loss': '0.02787', 'grad_norm': '0', 'learning_rate': '9.82e-06', 'num_tokens': '5813', 'completions/mean_length': '54.02', 'completions/min_length': '16.3', 'completions/max_length': '108.4', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '54.02', 'completions/min_terminated_length': '16.3', 'completions/max_terminated_length': '108.4', 'rewards/countdown_reward/mean': '0.5312', 'rewards/countdown_reward/std': '0.0625', 'reward': '0.5312', 'reward_std': '0.0625', 'frac_reward_zero_std': '0.9', 'entropy': '0.3275', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'cli

Generating train split: 327680 examples [00:03, 96810.60 examples/s] 
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 2503.48it/s, Materializing param=model.norm.weight]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 500/500 [56:47<00:00,  6.81s/it]


In [6]:
import os
import json
import pandas as pd

log_dir = "/content/drive/MyDrive/tinyzero-lora-grpo/outputs/grpo_from_sft"
samples_path = os.path.join(log_dir, "samples.json")

print(f"--- 目录文件列表 ---")
if os.path.exists(log_dir):
    display(os.listdir(log_dir))
else:
    print("目录尚未创建，请检查训练是否成功启动。")

print(f"\n--- 最新训练样本 (来自 samples.json) ---")
if os.path.exists(samples_path):
    with open(samples_path, 'r') as f:
        data = json.load(f)
    # 转为 DataFrame 方便查看最后几行
    df = pd.DataFrame(data)
    display(df.tail(4))
else:
    print("尚未生成 samples.json，可能训练步数还没达到设定的 log_every 步数。")

--- 目录文件列表 ---


['checkpoint-100',
 'checkpoint-200',
 'checkpoint-300',
 'checkpoint-400',
 'README.md',
 'samples.json',
 'checkpoint-500',
 'final_adapter']


--- 最新训练样本 (来自 samples.json) ---


,step,sample_idx,prompt,completion,parsed_answer,reward_breakdown,solved
36,500,0,You are solving a Countdown arithmetic puzzle....,<think>\n</think>\n<answer>\n</answer>\n\n<thi...,None,"{'format': 0.5, 'expr': 0.0, 'multiset': 0.0, ...",False
37,500,1,You are solving a Countdown arithmetic puzzle....,<think>\n</think>\n<answer>\n</answer>\n\n<thi...,None,"{'format': 0.5, 'expr': 0.0, 'multiset': 0.0, ...",False
38,500,2,You are solving a Countdown arithmetic puzzle....,<think>\n</think>\n<answer>\n</answer>\n\n<thi...,None,"{'format': 0.5, 'expr': 0.0, 'multiset': 0.0, ...",False
39,500,3,You are solving a Countdown arithmetic puzzle....,<think>\n</think>\n<answer>\n</answer>\n\n<thi...,None,"{'format': 0.5, 'expr': 0.0, 'multiset': 0.0, ...",False


In [4]:
path = "/content/drive/MyDrive/tinyzero-lora-grpo/src/parsing.py"
with open(path) as f:
    src = f.read()

old = '''    if not parsed.has_think_pair or parsed.answer_inner is None:
        return False
    if "=" in parsed.answer_inner:
        return False'''

new = '''    if not parsed.has_think_pair or not parsed.answer_inner:
        return False  # None 或空字符串都不给 format 分
    if "=" in parsed.answer_inner:
        return False'''

src = src.replace(old, new)
with open(path, "w") as f:
    f.write(src)
print("Done")


Done


In [16]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/train.py \
    --init_adapter outputs/sft_adapter/final_adapter \
    --max_steps 500 \
    --report_to none \
    --output_dir outputs/grpo_from_sft_v2

Process is terminated.


In [14]:
!sed -i 's/row "prompt"\])/row["prompt"])/g' /content/drive/MyDrive/tinyzero-lora-grpo/scripts/train.py

In [15]:
!sed -n '45,50p' /content/drive/MyDrive/tinyzero-lora-grpo/scripts/train.py

        for i in range(n):
            row = self.dataset[i]
            prompts.append(row["prompt"])
            nums_list.append(row["nums"])
            targets.append(row["target"])
        completions = []


In [9]:
path = "/content/drive/MyDrive/tinyzero-lora-grpo/scripts/train.py"
with open(path) as f:
    src = f.read()

src = src.replace(
    "init_model = PeftModel.from_pretrained(base, str(args.init_adapter))",
    "init_model = PeftModel.from_pretrained(base, str(args.init_adapter), is_trainable=True)"
)
with open(path, "w") as f:
    f.write(src)
print("Done")

Done


In [17]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/train.py \
    --init_adapter outputs/sft_adapter/final_adapter \
    --max_steps 500 \
    --report_to none \
    --output_dir outputs/grpo_from_sft_v3

Parameters: total=1,544,803,840 trainable=1,089,536 (0.0705% trainable)
{'loss': '0.03225', 'grad_norm': '1.1', 'learning_rate': '9.82e-06', 'num_tokens': '7438', 'completions/mean_length': '94.65', 'completions/min_length': '27', 'completions/max_length': '204.3', 'completions/clipped_ratio': '0.025', 'completions/mean_terminated_length': '83.69', 'completions/min_terminated_length': '27', 'completions/max_terminated_length': '167.4', 'rewards/countdown_reward/mean': '0.95', 'rewards/countdown_reward/std': '0.676', 'reward': '0.95', 'reward_std': '0.676', 'frac_reward_zero_std': '0', 'entropy': '1.14', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '9.025', 'epoch': '3.052e-05'}
{'loss': '-0.02645', 'grad_norm': '0.8193', 'learning_rate': '9.62e-06', 'num_tokens': '1.645e+04', 'completions/mean_length': '133.9', 'completions/min_length': '43.2', 'completions/max_length': '286.4

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 432.95it/s, Materializing param=model.norm.weight]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
100%|██████████| 500/500 [1:12:05<00:00,  8.65s/it]


In [19]:
%%bash
cd /content/drive/MyDrive/tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/eval.py \
    --model_id Qwen/Qwen2.5-1.5B \
    --adapter outputs/grpo_from_sft_v3/final_adapter \
    --data data/processed/test.parquet \
    --out outputs/eval_grpo_sft_v3.json \
    --max_samples 200

{
  "n": 200,
  "format_rate": 0.995,
  "valid_expr_rate": 0.995,
  "solve_rate": 0.01
}


Generating train split: 1024 examples [00:00, 164646.45 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
eval final_adapter: 100%|██████████| 200/200 [07:27<00:00,  2.24s/it]


In [20]:
content = '''#!/usr/bin/env python3
"""Debug reward: print input format to verify TRL reward_funcs signature."""

def debug_reward(prompts, completions, **kwargs):
    print("prompts[0]:", prompts[0][:80])
    print("completions[0]:", completions[0][:80])
    print("kwargs keys:", list(kwargs.keys()))
    return [0.0] * len(completions)
'''
with open("/content/drive/MyDrive/tinyzero-lora-grpo/scripts/debug_reward.py", "w") as f:
    f.write(content)
print("Done")

Done


In [22]:
readme = """# TinyZero Countdown — LoRA + GRPO

Reproducing the TinyZero Countdown task, replacing full fine-tuning with LoRA (PEFT) + GRPO (TRL).

## Project Goal

- Verify whether LoRA + GRPO can enable small models to learn structured reasoning
- Demonstrate parameter efficiency: only ~0.07% of parameters are updated vs full fine-tuning

## Environment

- Python 3.10+
- CUDA GPU (tested on Google Colab A100)
- Key dependencies: `torch>=2.2`, `transformers>=4.46`, `trl==0.29.1`, `peft==0.18.1`

```bash
pip install -r requirements.txt
```

## Data Preparation

cd tinyzero-lora-grpo
export PYTHONPATH=$(pwd)
python scripts/prepare_data.py

Generates data/processed/train.parquet (327,680 rows) and test.parquet (1,024 rows), matching TinyZero's exact index-based split.

## Training

Step 1 — Pipeline smoke test (dummy reward):
python scripts/train.py --dummy_reward --max_steps 5 --report_to none --output_dir outputs/smoke

Step 2 — SFT warm-up (teach output format):
python scripts/sft.py --num_examples 120 --epochs 3 --output_dir outputs/sft_adapter

Step 3 — GRPO from SFT adapter:
python scripts/train.py \\
    --init_adapter outputs/sft_adapter/final_adapter \\
    --max_steps 500 \\
    --report_to none \\
    --output_dir outputs/grpo_from_sft

## Evaluation

Base model baseline:
python scripts/eval.py \\
    --model_id Qwen/Qwen2.5-1.5B \\
    --data data/processed/test.parquet \\
    --out outputs/eval_base.json \\
    --max_samples 200

Trained model:
python scripts/eval.py \\
    --model_id Qwen/Qwen2.5-1.5B \\
    --adapter outputs/grpo_from_sft/final_adapter \\
    --data data/processed/test.parquet \\
    --out outputs/eval_lora.json \\
    --max_samples 200

## Results

| Model | format_rate | valid_expr_rate | solve_rate |
|-------|-------------|-----------------|------------|
| Base model (Qwen2.5-1.5B) | 0.0 | 0.0 | 0.0 |
| SFT + GRPO 500 steps | 0.995 | 0.995 | 0.01 |

LoRA parameter efficiency:
- Total parameters: 1,544,803,840
- Trainable parameters: 1,089,536
- Trainable %: 0.0705%

## Reward Function (Layered)

1. Format: <think>...</think><answer>...</answer>  (+0.5)
2. Valid arithmetic expression (AST-based)         (+0.5)
3. Correct number usage (multiset match)           (+0.75)
4. Result equals target                            (+3.0)
"""

import os
with open("/content/drive/MyDrive/tinyzero-lora-grpo/README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print("Done")

Done
